# The Standard Model in FeynPy (`SM.fr` style)

This notebook is the FeynPy analogue of the FeynRules model file `SM.fr`.

It is written in exactly the same style as `list_lagrangians.ipynb`: we only use
the public `feynpy` primitives (`Field`, `GaugeGroup`, `Parameter`, `Model`,
`CovD`, `FieldStrength`, `Gamma`, `FieldTransformation`, ...). No new classes or
frameworks are introduced.

The layout follows `SM.fr` section by section:

1. Information
2. Indices
3. Parameters
4. Particle classes (fields)
5. Gauge groups
6. Lagrangian sectors (`LGauge`, `LFermions`, `LHiggs`, `LYukawa`)
7. Field `Definitions` — rewrite gauge-basis fields into physical ones
8. Build + Feynman rules

**Workflow (same as FeynRules):** write `LSM` with unbroken fields (`B`, `Wi`,
`Phi`, `LL`, `QL`, ...), then apply the `Definitions` from `SM.fr` to obtain
vertices with `A`, `Z`, `W`, `H`, `e`, `u`, `d`, ...

## Setup

In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (ROOT, SRC):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from symbolica import Expression, S

from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    CovD,
    Field,
    FieldStrength,
    FieldTransformation,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostLagrangian,
    LORENTZ_INDEX,
    Model,
    Parameter,
    PartialD,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    flavor_index,
    replacement,
)
from symbolic.spenso_structures import (
    chiral_projector_left,
    chiral_projector_right,
    gauge_generator,
    structure_constant,
    weak_eps2,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I
from symbolic.tensor_canonicalization import canonize_full

num = Expression.num
HALF = num(1) / num(2)
INV_SQRT2 = HALF ** HALF

ANSI = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    print(clean(result))
    print()

## 1. Information

The FeynRules `M$ModelName` / `M$Information` header, as plain Python metadata.

In [2]:
ModelName = "Standard Model"
Information = {
    "Authors": ("N. Christensen", "C. Duhr", "B. Fuks"),
    "URLs": "http://feynrules.phys.ucl.ac.be/view/Main/StandardModel",
}
FeynmanGauge = True

print(ModelName)
for key, value in Information.items():
    print(f"  {key}: {value}")

Standard Model
  Authors: ('N. Christensen', 'C. Duhr', 'B. Fuks')
  URLs: http://feynrules.phys.ucl.ac.be/view/Main/StandardModel


## 2. Indices

`SM.fr` declares `Generation`, `SU2W` (weak adjoint), `SU2D` (weak doublet),
`Colour`, and `Gluon`. The colour, weak and Lorentz/spinor index types ship with
FeynPy; only the flavour index has to be declared, exactly like in
`list_lagrangians.ipynb`.

In [3]:
Generation = flavor_index("Generation", 3, prefix="fl")
Generation

IndexType(name='Generation', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='generation', dimension=3, is_flavor=True, prefix='fl')

## 3. Parameters

The `M$Parameters` block: gauge couplings `g1, g2, g3`, the Higgs `vev` and
quartic `lam`, the derived electroweak quantities `sw, cw, ee`, the masses, the
Yukawa matrices and the (Cabibbo-only) `CKM` matrix.

`Parameter(..., value=...)` mirrors a FeynRules `Internal` parameter
(`Definitions`), while `Parameter(..., indices=..., components=...)` mirrors an
indexed parameter with an explicit `Value` table.

In [4]:
g1 = Parameter("g1")
g2 = Parameter("g2")
g3 = Parameter("g3")
lam = Parameter("lam")
vev = Parameter("vev")

gz = (g1.symbol**2 + g2.symbol**2) ** HALF
sw = Parameter("sw", value=g1.symbol / gz)            # sin(theta_W) = g1 / sqrt(g1^2+g2^2)
cw = Parameter("cw", value=g2.symbol / gz)            # cos(theta_W) = g2 / sqrt(g1^2+g2^2)
ee = Parameter("ee", value=g1.symbol * g2.symbol / gz)  # e = g1 g2 / sqrt(g1^2+g2^2)

MW = Parameter("MW", value=g2.symbol * vev.symbol / 2)
MZ = Parameter("MZ", value=gz * vev.symbol / 2)
MH = Parameter("MH", value=(2 * lam.symbol * vev.symbol**2) ** HALF)

# R_xi gauge parameters (kept symbolic)
xiA = Parameter("xiA", internal=False, value=S("xiA"))
xiZ = Parameter("xiZ", internal=False, value=S("xiZ"))
xiW = Parameter("xiW", internal=False, value=S("xiW"))
xiG = Parameter("xiG", internal=False, value=S("xiG"))

cabi = S("cabi")
cos, sin = S("cos")(cabi), S("sin")(cabi)


def diagonal(prefix):
    """Diagonal 3x3 Yukawa table, e.g. yu[i,i] -> yu1, yu2, yu3."""
    return {(r, c): (S(f"{prefix}{r}") if r == c else num(0))
            for r in range(1, 4) for c in range(1, 4)}


def mass_param(name, prefix):
    """Diagonal mass M[i] = vev * y[i] / sqrt(2)."""
    return Parameter(name, indices=(Generation,),
                     components={(i,): INV_SQRT2 * vev.symbol * S(f"{prefix}{i}") for i in range(1, 4)})


ckm = {(1, 1): cos, (1, 2): sin, (1, 3): num(0),
       (2, 1): -sin, (2, 2): cos, (2, 3): num(0),
       (3, 1): num(0), (3, 2): num(0), (3, 3): num(1)}
ckm_T = {(c, r): v for (r, c), v in ckm.items()}

Mvl = Parameter("Mvl", indices=(Generation,), components={(i,): num(0) for i in range(1, 4)})
Ml = mass_param("Ml", "ye")
Mu = mass_param("Mu", "yu")
Md = mass_param("Md", "yd")

Yu = Parameter("Yu", indices=(Generation, Generation), complex_param=True, components=diagonal("yu"))
YuDag = Parameter("YuDag", indices=(Generation, Generation), complex_param=True, components=diagonal("yu"))
Yd = Parameter("Yd", indices=(Generation, Generation), complex_param=True, components=diagonal("yd"))
YdDag = Parameter("YdDag", indices=(Generation, Generation), complex_param=True, components=diagonal("yd"))
Ye = Parameter("Ye", indices=(Generation, Generation), complex_param=True, components=diagonal("ye"))
YeDag = Parameter("YeDag", indices=(Generation, Generation), complex_param=True, components=diagonal("ye"))
CKM = Parameter("CKM", indices=(Generation, Generation), complex_param=True, components=ckm, unitary_partner="CKMDag")
CKMDag = Parameter("CKMDag", indices=(Generation, Generation), complex_param=True, components=ckm_T, unitary_partner="CKM")

all_parameters = (g1, g2, g3, lam, vev, sw, cw, ee, MW, MZ, MH, xiA, xiZ, xiW, xiG,
                  Mvl, Ml, Mu, Md, Yu, YuDag, Yd, YdDag, Ye, YeDag, CKM, CKMDag)
print("sin(theta_W) =", sw.value)
print("e            =", ee.value)
print("MW           =", MW.value)

sin(theta_W) = g1/(g1^2+g2^2)^(1/2)
e            = g1*g2/(g1^2+g2^2)^(1/2)
MW           = 1/2*g2*vev


## 4. Particle classes (field declarations)

This is the `M$ClassesDescription` block. As in `list_lagrangians.ipynb`, every
field is one `Field(...)` declaration.

We declare two groups, exactly like `SM.fr`:

- **Gauge-basis (unphysical) fields** `B, Wi, G, LL, lR, QL, uR, dR, Phi` and the
  gauge ghosts. These carry the gauge quantum numbers (`Y`, weak doublet index,
  colour) and are what the Lagrangian is written in.
- **Physical (mass-eigenstate) fields** `A, Z, W, H, G0, GP` and the fermion
  flavour classes `vl, l, uq, dq`.

In [5]:
def ghost(name, *, indices=(), ghost_of=None, mass=None, qn=None):
    return Field(name, spin=0, kind="ghost", self_conjugate=False, indices=tuple(indices),
                 ghost_of=ghost_of, symbol=S(name), conjugate_symbol=S(f"{name}bar"),
                 mass=mass, quantum_numbers=dict(qn or {}))


# --- Gauge-basis (unphysical) fields ---
B = Field("B", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,))
Wi = Field("Wi", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
G = Field("G", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
ghB = ghost("ghB", ghost_of=B)
ghWi = ghost("ghWi", indices=(WEAK_ADJ_INDEX,), ghost_of=Wi)
ghG = ghost("ghG", indices=(COLOR_ADJ_INDEX,), ghost_of=G)

LL = Field("LL", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, WEAK_FUND_INDEX, Generation), quantum_numbers={"Y": -HALF})
lR = Field("lR", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, Generation), quantum_numbers={"Y": -num(1)})
QL = Field("QL", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, WEAK_FUND_INDEX, Generation, COLOR_FUND_INDEX),
           quantum_numbers={"Y": num(1) / num(6)})
uR = Field("uR", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX), quantum_numbers={"Y": num(2) / num(3)})
dR = Field("dR", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX), quantum_numbers={"Y": -(num(1) / num(3))})
Phi = Field("Phi", spin=0, self_conjugate=False, symbol=S("Phi"), conjugate_symbol=S("Phibar"),
            indices=(WEAK_FUND_INDEX,), quantum_numbers={"Y": HALF})

# --- Physical (mass-eigenstate) fields ---
W = Field("W", spin=1, self_conjugate=False, conjugate_symbol=S("Wbar"), indices=(LORENTZ_INDEX,),
          mass=MW.symbol, quantum_numbers={"Q": num(1)})
Z = Field("Z", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,), mass=MZ.symbol, quantum_numbers={"Q": num(0)})
A = Field("A", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,), mass=num(0), quantum_numbers={"Q": num(0)})
H = Field("H", spin=0, self_conjugate=True, mass=MH.symbol, quantum_numbers={"Q": num(0)})
G0 = Field("G0", spin=0, self_conjugate=True, mass=MZ.symbol, quantum_numbers={"Q": num(0)}, goldstone_of=Z)
GP = Field("GP", spin=0, self_conjugate=False, conjugate_symbol=S("GM"), mass=MW.symbol,
           quantum_numbers={"Q": num(1)}, goldstone_of=W)
ghA = ghost("ghA", ghost_of=A, mass=num(0), qn={"GhostNumber": num(1)})
ghZ = ghost("ghZ", ghost_of=Z, mass=MZ.symbol, qn={"GhostNumber": num(1)})
ghWp = ghost("ghWp", ghost_of=W, mass=MW.symbol, qn={"GhostNumber": num(1), "Q": num(1)})
ghWm = ghost("ghWm", ghost_of=W, mass=MW.symbol, qn={"GhostNumber": num(1), "Q": -num(1)})

vl = Field("vl", spin=Fraction(1, 2), self_conjugate=False, indices=(SPINOR_INDEX, Generation),
           mass=Mvl, quantum_numbers={"Q": num(0), "LeptonNumber": num(1)},
           flavor_index=Generation, class_members=("ve", "vm", "vt"))
l = Field("l", spin=Fraction(1, 2), self_conjugate=False, indices=(SPINOR_INDEX, Generation),
          mass=Ml, quantum_numbers={"Q": -num(1), "LeptonNumber": num(1)},
          flavor_index=Generation, class_members=("e", "mu", "ta"))
uq = Field("uq", spin=Fraction(1, 2), self_conjugate=False, indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX),
           mass=Mu, quantum_numbers={"Q": num(2) / num(3)}, flavor_index=Generation, class_members=("u", "c", "t"))
dq = Field("dq", spin=Fraction(1, 2), self_conjugate=False, indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX),
           mass=Md, quantum_numbers={"Q": -(num(1) / num(3))}, flavor_index=Generation, class_members=("d", "s", "b"))

print("charged leptons:", [m.name for m in l.class_members])
print("up quarks:      ", [m.name for m in uq.class_members])
print("down quarks:    ", [m.name for m in dq.class_members])

charged leptons: ['e', 'mu', 'ta']
up quarks:       ['u', 'c', 't']
down quarks:     ['d', 's', 'b']


## 5. Gauge groups

The `M$GaugeGroups` block: `U1Y`, `SU2L`, `SU3C`. Each `GaugeGroup` points at its
gauge boson, ghost, coupling and representation, just like `list_lagrangians.ipynb`.

In [6]:
U1Y = GaugeGroup("U1Y", abelian=True, coupling=g1, gauge_boson=B, ghost_field=ghB, charge="Y")
SU2L = GaugeGroup(
    "SU2L", abelian=False, coupling=g2, gauge_boson=Wi, ghost_field=ghWi,
    structure_constant=weak_structure_constant,
    representations=(GaugeRepresentation(index=WEAK_FUND_INDEX, generator_builder=weak_gauge_generator, name="doublet"),),
)
SU3C = GaugeGroup(
    "SU3C", abelian=False, coupling=g3, gauge_boson=G, ghost_field=ghG,
    structure_constant=structure_constant,
    representations=(GaugeRepresentation(index=COLOR_FUND_INDEX, generator_builder=gauge_generator, name="fundamental"),),
)
gauge_groups = (U1Y, SU2L, SU3C)
gauge_groups

(GaugeGroup(name='U1Y', abelian=True, coupling=Parameter(name='g1', symbol=g1, indices=(), complex_param=False, internal=True, value=None, components={}, allow_summation=None, unitary_partner=None), gauge_boson=Field(name='B', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=4, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=B, conjugate_symbol=None, mass=None, quantum_numbers={}, ghost_of=None, goldstone_of=None, flavor_index=None, class_members=()), ghost_field=Field(name='ghB', spin=0, self_conjugate=False, indices=(), kind='ghost', statistics='fermion', symbol=ghB, conjugate_symbol=ghBbar, mass=None, quantum_numbers={}, ghost_of=Field(name='B', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=4, is_flavor=False, prefix='mu'),), k

## 6. Lagrangian sectors

The `LGauge`, `LFermions`, `LHiggs` and `LYukawa` sectors of `SM.fr`, written in
the **gauge basis** with `feynpy` operators (`FieldStrength`, `CovD`, `Gamma`).
These are the same building blocks used in `list_lagrangians.ipynb`, just
assembled into the full Standard Model.

In [7]:
mu, nu = S("mu"), S("nu")
aW, aC = S("aW"), S("aC")
sp, ii, jj, cc = S("sp"), S("ii"), S("jj"), S("cc")
f1, f2, f3 = S("ff1"), S("ff2"), S("ff3")

# LGauge: -1/4 B.B - 1/4 Wi.Wi - 1/4 G.G
LGauge = (
    -num(1) / num(4) * FieldStrength(U1Y, mu, nu) * FieldStrength(U1Y, mu, nu)
    - num(1) / num(4) * FieldStrength(SU2L, mu, nu, aW) * FieldStrength(SU2L, mu, nu, aW)
    - num(1) / num(4) * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)
)

# LFermions: i psibar gamma^mu D_mu psi
LFermions = (
    I * QL.bar * Gamma(mu) * CovD(QL, mu)
    + I * LL.bar * Gamma(mu) * CovD(LL, mu)
    + I * uR.bar * Gamma(mu) * CovD(uR, mu)
    + I * dR.bar * Gamma(mu) * CovD(dR, mu)
    + I * lR.bar * Gamma(mu) * CovD(lR, mu)
)

# LHiggs: |D Phi|^2 + muH^2 |Phi|^2 - lam |Phi|^4
LHiggs = (
    CovD(Phi.bar, mu) * CovD(Phi, mu)
    + lam * vev**2 * Phi.bar * Phi
    - lam * Phi.bar * Phi * Phi.bar * Phi
)

# LYukawa: down, lepton and up Yukawa terms + hermitian conjugates
LYukawa = (
    -Yd(f2, f3) * CKM(f1, f2) * QL.bar(sp, ii, f1, cc) * dR(sp, f3, cc) * Phi(ii)
    - Ye(f1, f3) * LL.bar(sp, ii, f1) * lR(sp, f3) * Phi(ii)
    - Yu(f1, f2) * QL.bar(sp, ii, f1, cc) * uR(sp, f2, cc) * Phi.bar(jj) * weak_eps2(ii, jj)
    - YdDag(f3, f2) * CKMDag(f2, f1) * Phi.bar(ii) * dR.bar(sp, f3, cc) * QL(sp, ii, f1, cc)
    - YeDag(f3, f1) * Phi.bar(ii) * lR.bar(sp, f3) * LL(sp, ii, f1)
    - YuDag(f2, f1) * weak_eps2(ii, jj) * Phi(jj) * uR.bar(sp, f2, cc) * QL(sp, ii, f1, cc)
)

# Manual R_xi gauge-fixing and ghost sectors
LGaugeFixing = (
    GaugeFixing(U1Y, xi=xiA.value)
    + GaugeFixing(SU2L, xi=xiW.value)
    + GaugeFixing(SU3C, xi=xiG.value)
)
LGhost = (
    PartialD(ghB.bar, mu) * PartialD(ghB, mu)
    + GhostLagrangian(SU2L)
    + GhostLagrangian(SU3C)
)

LSM = LGauge + LFermions + LHiggs + LYukawa + (LGaugeFixing + LGhost if FeynmanGauge else 0)
show("LGauge", LGauge)
show("LHiggs", LHiggs)
show("LGaugeFixing", LGaugeFixing)
show("LGhost", LGhost)

LGauge
-1/4 * FieldStrength(U1Y, mu, nu) * FieldStrength(U1Y, mu, nu) + -1/4 * FieldStrength(SU2L, mu, nu, aW) * FieldStrength(SU2L, mu, nu, aW) + -1/4 * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)

LHiggs
CovD(Phi.bar, mu) * CovD(Phi, mu) + lam*vev^2 * Phi.bar * Phi + -lam * Phi.bar * Phi * Phi.bar * Phi

LGaugeFixing
GaugeFixing(U1Y, xi=xiA) + GaugeFixing(SU2L, xi=xiW) + GaugeFixing(SU3C, xi=xiG)

LGhost
PartialD(ghB.bar, mu) * PartialD(ghB, mu) + GhostLagrangian(SU2L) + GhostLagrangian(SU3C)



## 7. Field `Definitions` (symmetry breaking)

**What is this?** In `SM.fr`, section 6 writes the Lagrangian with *unphysical*
fields (`B`, `Wi`, `Phi`, `LL`, `QL`, ...). Section 7 is not more Lagrangian —
it is the list of **`Definitions`** that rewrite those fields into the *physical*
particles (`A`, `Z`, `W`, `H`, `e`, `u`, `d`, ...).

In FeynPy each definition is one `FieldTransformation`: “replace occurrences of
`B` by `-sw*Z + cw*A`”, etc.

**Is it necessary?** Yes, if you want the `SM.fr` workflow (simple gauge-basis
`LSM` → physical Feynman rules). Without this step you only get vertices in terms
of `B` and `Wi`, not `A` and `W`.

**Two parts:**
- **Bosons & Higgs (& ghosts)** — linear mixes and the vev shift; written directly below
  (same formulas as `SM.fr`).
- **Fermions** — `SM.fr` uses chiral projectors `ProjM` / `ProjP` in the
  definitions (`LL -> ProjM l`, ...). A projector is a *spinor matrix*, so it
  carries an extra spinor index that a plain `replacement(coeff, field)` has no
  slot for. One tiny helper, `fermion(source, target, Proj, ...)`, wires that
  spinor index (and the optional CKM rotation) so the rules below read just like
  `SM.fr`: `fermion(LL, l, ProjM)` is `LL -> ProjM l`.

In [8]:
# SM.fr-style chiral fermion definitions:  source -> Proj * target
# A projector is a spinor matrix, so (unlike the boson mixings above) it carries
# one extra spinor index. This single helper wires that spinor index and the
# optional CKM flavor rotation; the call sites below then read like `LL -> ProjM*l`.
ProjM, ProjP = chiral_projector_left, chiral_projector_right
_conj_proj = {ProjM: ProjP, ProjP: ProjM}

def fermion(source, target, proj, *, components=None, mixing=None):
    def make(conjugated):
        def build(ctx):
            labels = {idx: ctx.label(slot) for slot, idx in enumerate(ctx.occurrence.field.indices)}
            s_src = labels[SPINOR_INDEX]
            g_src = next(lbl for idx, lbl in labels.items() if idx.is_flavor)
            s_tgt = ctx.fresh(SPINOR_INDEX, "f")
            g_tgt = g_src
            coeff = num(1)
            if mixing is not None:
                rotate, rotate_dag = mixing
                g_idx = next(idx for idx in target.indices if idx.is_flavor)
                g_tgt = ctx.fresh(g_idx, "mix")
                coeff *= rotate_dag(g_tgt, g_src) if conjugated else rotate(g_src, g_tgt)
            P = _conj_proj[proj] if conjugated else proj
            coeff *= P(s_tgt, s_src) if conjugated else P(s_src, s_tgt)
            tgt_labels = {}
            for slot, idx in enumerate(target.indices):
                if idx == SPINOR_INDEX:
                    tgt_labels[slot] = s_tgt
                elif idx.is_flavor:
                    tgt_labels[slot] = g_tgt
                elif idx in labels:
                    tgt_labels[slot] = labels[idx]
            occ = target.occurrence(conjugated=conjugated, labels=target.pack_slot_labels(tgt_labels))
            return (replacement(coeff, occ),)
        return build
    return FieldTransformation(source, components=components or {},
                               builder=make(False), conjugate_builder=make(True))


swv, cwv = sw.value, cw.value
transformations = (
    # gauge-boson definitions
    FieldTransformation(B, terms=(replacement(-swv, Z), replacement(cwv, A))),
    FieldTransformation(Wi, components={1: 1}, terms=(replacement(INV_SQRT2, W.bar), replacement(INV_SQRT2, W))),
    FieldTransformation(Wi, components={1: 2}, terms=(replacement(INV_SQRT2 / I, W.bar), replacement(-INV_SQRT2 / I, W))),
    FieldTransformation(Wi, components={1: 3}, terms=(replacement(cwv, Z), replacement(swv, A))),

    # Higgs/Goldstone definitions
    FieldTransformation(Phi, components={0: 1}, terms=(replacement(-I, GP),)),
    FieldTransformation(Phi, components={0: 2}, terms=(
        replacement(vev.symbol * INV_SQRT2),
        replacement(INV_SQRT2, H),
        replacement(I * INV_SQRT2, G0),
    )),

    # ghost definitions
    FieldTransformation(ghB, terms=(replacement(-swv, ghZ), replacement(cwv, ghA))),
    FieldTransformation(ghWi, components={0: 1}, terms=(replacement(INV_SQRT2, ghWp), replacement(INV_SQRT2, ghWm))),
    FieldTransformation(ghWi, components={0: 2}, terms=(replacement(-INV_SQRT2 / I, ghWp), replacement(INV_SQRT2 / I, ghWm))),
    FieldTransformation(ghWi, components={0: 3}, terms=(replacement(cwv, ghZ), replacement(swv, ghA))),

    # fermion definitions:  source -> Proj * target   (SM.fr ProjM/ProjP)
    fermion(LL, vl, ProjM, components={1: 1}),
    fermion(LL, l,  ProjM, components={1: 2}),
    fermion(lR, l,  ProjP),
    fermion(QL, uq, ProjM, components={1: 1}),
    fermion(QL, dq, ProjM, components={1: 2}, mixing=(CKM, CKMDag)),
    fermion(uR, uq, ProjP),
    fermion(dR, dq, ProjP),
)
print(len(transformations), "field definitions declared")

17 field definitions declared


### Weak-index unfolding (technical)

`Wi` and `Phi` carry a symbolic SU(2) index (`1`, `2`, or `3`). Before the
definitions can act on `Wi[1]`, `Wi[2]`, `Wi[3]`, FeynPy must expand that index
to explicit components (FeynRules: `ExpandIndices[..., FlavorExpand->SU2W]`).
The numeric Pauli/epsilon table below is written explicitly (no helper import).

In [9]:
def concrete(builder, *values):
    labels = tuple(S(f"component_{i}") for i in range(len(values)))
    expr = builder(*labels)
    for lbl, val in zip(labels, values):
        expr = expr.replace(lbl, num(val))
    return expr


weak_tensor_components = {}
pauli_over_two = {
    (1, 1, 1): 0, (1, 1, 2): HALF, (1, 2, 1): HALF, (1, 2, 2): 0,
    (2, 1, 1): 0, (2, 1, 2): -I * HALF, (2, 2, 1): I * HALF, (2, 2, 2): 0,
    (3, 1, 1): HALF, (3, 1, 2): 0, (3, 2, 1): 0, (3, 2, 2): -HALF,
}
for labels, value in pauli_over_two.items():
    weak_tensor_components[concrete(weak_gauge_generator, *labels)] = value
for a in range(1, 4):
    for b in range(1, 4):
        for c in range(1, 4):
            if len({a, b, c}) < 3:
                v = 0
            else:
                vals = (a, b, c)
                inv = sum(x > y for i, x in enumerate(vals) for y in vals[i + 1:])
                v = -1 if inv % 2 else 1
            weak_tensor_components[concrete(weak_structure_constant, a, b, c)] = num(v)
for a in range(1, 3):
    for b in range(1, 3):
        v = 1 if (a, b) == (1, 2) else -1 if (a, b) == (2, 1) else 0
        weak_tensor_components[concrete(weak_eps2, a, b)] = num(v)
print(len(weak_tensor_components), "SU(2) tensor components")

43 SU(2) tensor components


## 8. Build and extract Feynman rules

Three engine steps (automatic in FeynRules after loading `SM.fr`):

1. **Compile** the gauge-basis `LSM` (`CovD` and `FieldStrength` are expanded).
2. **Unfold** weak indices (`expand_index_components`) so `Wi[1]` is a real slot.
3. **Apply** the field `Definitions` (`transform_fields`) to get `A`, `W`, `H`, ...

In [10]:
source_fields = (LL, lR, QL, uR, dR, Phi, B, Wi, G, ghB, ghWi, ghG)
source_model = Model(
    name="Standard Model (gauge basis)",
    gauge_groups=gauge_groups,
    fields=source_fields,
    parameters=all_parameters,
    lagrangian_decl=LSM,
)

real_symbols = (g1, g2, g3, lam, vev, MW, MZ, MH, sw.value, cw.value, ee.value)

L = (
    source_model.lagrangian()
    .expand_index_components(WEAK_FUND_INDEX, WEAK_ADJ_INDEX, tensor_components=weak_tensor_components)
    .transform_fields(*transformations, repeat=False, real_symbols=real_symbols)
    .simplify_parameter_identities()
)

# Physical-basis model used to ask for Feynman rules.
physical_fields = (vl, l, uq, dq, H, G0, GP, W, Z, A, G) + ((ghA, ghZ, ghWp, ghWm, ghG) if FeynmanGauge else ())
sm = Model(name=ModelName, fields=physical_fields, parameters=all_parameters)
sm._compiled_lagrangian = L

print("gauge-basis compiled terms:", len(source_model.lagrangian().terms))
print("broken-phase terms:        ", len(L.terms))
print("vertex signatures:         ", len(L.vertex_signatures()))

gauge-basis compiled terms: 62
broken-phase terms:         294
vertex signatures:          115


## 9. Feynman rules

The physical-basis vertices, extracted exactly as in `list_lagrangians.ipynb`
with `L.feynman_rule(...)`.

In [11]:
def cf(expr):
    return canonize_full(expr, infer_indices=True, field_heads=tuple(sm.fields), run_color=False)


show("Gamma(W-, W+, A)", L.feynman_rule(W.bar, W, A, simplify=True))
show("Gamma(W-, W+, Z)", L.feynman_rule(W.bar, W, Z, simplify=True))
show("Gamma(W-, W+, A, A)", L.feynman_rule(W.bar, W, A, A, simplify=True))
show("Gamma(H, W-, W+)", L.feynman_rule(H, W.bar, W, simplify=True))
show("Gamma(H, Z, Z)", L.feynman_rule(H, Z, Z, simplify=True))
show("Gamma(H, H, H)", L.feynman_rule(H, H, H, simplify=True))
show("Gamma(lbar, l, A)  (lepton QED current)", cf(L.feynman_rule(l.bar, l, A, simplify=True)))
show("Gamma(ubar, d, W+)  (CKM charged current)", L.feynman_rule(uq.bar, dq, W, simplify=True))
show("Gamma(ubar, u, G)   (QCD vertex)", L.feynman_rule(uq.bar, uq, G, simplify=True))

Gamma(W-, W+, A)
1𝑖*g1*g2*g(mink(4, mu1),mink(4, mu2))*pcomp(q1,mu3)/(g1^2+g2^2)^(1/2)-1𝑖*g1*g2*g(mink(4, mu1),mink(4, mu2))*pcomp(q2,mu3)/(g1^2+g2^2)^(1/2)-1𝑖*g1*g2*g(mink(4, mu1),mink(4, mu3))*pcomp(q1,mu2)/(g1^2+g2^2)^(1/2)+1𝑖*g1*g2*g(mink(4, mu1),mink(4, mu3))*pcomp(q3,mu2)/(g1^2+g2^2)^(1/2)+1𝑖*g1*g2*g(mink(4, mu2),mink(4, mu3))*pcomp(q2,mu1)/(g1^2+g2^2)^(1/2)-1𝑖*g1*g2*g(mink(4, mu2),mink(4, mu3))*pcomp(q3,mu1)/(g1^2+g2^2)^(1/2)

Gamma(W-, W+, Z)
1𝑖*g2^2*g(mink(4, mu1),mink(4, mu2))*pcomp(q1,mu3)/(g1^2+g2^2)^(1/2)-1𝑖*g2^2*g(mink(4, mu1),mink(4, mu2))*pcomp(q2,mu3)/(g1^2+g2^2)^(1/2)-1𝑖*g2^2*g(mink(4, mu1),mink(4, mu3))*pcomp(q1,mu2)/(g1^2+g2^2)^(1/2)+1𝑖*g2^2*g(mink(4, mu1),mink(4, mu3))*pcomp(q3,mu2)/(g1^2+g2^2)^(1/2)+1𝑖*g2^2*g(mink(4, mu2),mink(4, mu3))*pcomp(q2,mu1)/(g1^2+g2^2)^(1/2)-1𝑖*g2^2*g(mink(4, mu2),mink(4, mu3))*pcomp(q3,mu1)/(g1^2+g2^2)^(1/2)

Gamma(W-, W+, A, A)
-2𝑖*g1^2*g2^2*g(mink(4, mu1),mink(4, mu2))*g(mink(4, mu3),mink(4, mu4))/(g1^2+g2^2)+1𝑖*g1^2*g2^2*g(mink(4, mu1

## 10. Sanity checks (fully standalone)

No `build_standard_model()` is used below. We perform internal checks that the
manual construction worked:

- no source-basis fields (`B`, `Wi`, `Phi`, `LL`, `QL`, ...) survive in `L`;
- representative EW/QCD/ghost vertices are non-zero;
- gauge-fixing sector contributes longitudinal terms.

In [12]:
def canon(expr):
    return expr.expand().to_canonical_string()


source_basis_names = {"B", "Wi", "Phi", "LL", "QL", "lR", "uR", "dR", "ghB", "ghWi"}
survivors = sorted({occ.field.name for term in L.terms for occ in term.fields if occ.field.name in source_basis_names})
print("source-basis survivors:", survivors)

checks = {
    "WWA": L.feynman_rule(W.bar, W, A, simplify=True),
    "HWW": L.feynman_rule(H, W.bar, W, simplify=True),
    "llA": cf(L.feynman_rule(l.bar, l, A, simplify=True)),
    "udW": L.feynman_rule(uq.bar, dq, W, simplify=True),
    "uuG": L.feynman_rule(uq.bar, uq, G, simplify=True),
    "ghWp_kin": L.feynman_rule(ghWp.bar, ghWp, simplify=True),
    "ghWpWghA": L.feynman_rule(ghWp.bar, W, ghA, simplify=True),
    "AA_2pt": L.feynman_rule(A, A, simplify=True),
}

for key, expr in checks.items():
    print(f"{key:9} nonzero:", canon(expr) != "0")

source-basis survivors: []
WWA       nonzero: True
HWW       nonzero: True
llA       nonzero: True
udW       nonzero: True
uuG       nonzero: True
ghWp_kin  nonzero: True
ghWpWghA  nonzero: True
AA_2pt    nonzero: True


## Summary

This notebook defines the Standard Model the way a FeynPy user is meant to:

- one `Field(...)` per particle class (gauge basis + physical), one
  `GaugeGroup(...)` per gauge factor, one `Parameter(...)` per parameter;
- the Lagrangian written in the unbroken gauge basis as `LGauge + LFermions +
  LHiggs + LYukawa`, using only `FieldStrength`, `CovD`, `Gamma` and the Yukawa
  contractions;
- the spontaneous symmetry breaking expressed declaratively as the field
  `Definitions` (`FieldTransformation`s), exactly mirroring `SM.fr`.

The engine then unfolds the weak indices, applies the definitions and returns the
physical Feynman rules, which match the FeynRules `SM.fr` reference (last cell).

This notebook is now fully standalone: it manually defines

- `LGaugeFixing` via `GaugeFixing(...)` declarations;
- `LGhost` via an explicit abelian ghost kinetic term plus `GhostLagrangian(...)`
  for the non-abelian groups;
- ghost field transformations (`ghB`, `ghWi`) alongside boson/Higgs/fermion
  transformations.

So the full workflow is pure `feynpy` + explicit notebook declarations, without
calling `build_standard_model()`. 